# 05 - Model Interpretation

This notebook examines the features contributing to the final classification models.

Feature importance was recorded across the cross-validation folds in Notebook 03. Because feature selection is performed independently within each training fold, the selected features can differ between folds.

The analysis therefore considers both:

- the frequency with which a feature was selected across folds
- the magnitude of its model-specific importance when selected

This provides a more robust assessment of feature stability than interpreting a single fitted model.

### Import Required Libraries

In [11]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Load Cross-Validation Feature Importance

Feature importance was extracted from each fitted model within each cross-validation fold in Notebook 03.

The resulting table records:

- the model and molecular dataset
- the cross-validation fold
- the selected feature
- the corresponding model-specific importance

These fold-level results are used to assess feature stability across the cross-validation procedure.

In [12]:
# Load fold-level feature importance generated in Notebook 03.
feature_importance_df = pd.read_csv(
    "../Results/Tables/Feature_Importance/Feature_Importance_By_Fold.csv"
)

# Inspect the structure of the feature-importance table.
feature_importance_df.head()

,Model,Fold,Feature,Importance
0,LogReg_Gene,1,11715380_s_at,0.139162
1,LogReg_Gene,1,11715670_a_at,0.194029
2,LogReg_Gene,1,11715671_x_at,0.189578
3,LogReg_Gene,1,11715946_a_at,0.146438
4,LogReg_Gene,1,11716395_a_at,0.125511


### Validate Feature-Importance Data

Confirm that the expected models, folds, and features are present before aggregation.

In [13]:
# Check the models represented in the feature-importance results.
print("Models:")
print(feature_importance_df["Model"].unique())

# Check the cross-validation folds represented in the results.
print("\nFolds:")
print(sorted(feature_importance_df["Fold"].unique()))

# Check the number of records in the table.
print("\nNumber of feature-importance records:")
print(len(feature_importance_df))

Models:
['LogReg_Gene' 'RF_Gene' 'LogReg_miRNA' 'RF_miRNA']

Folds:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Number of feature-importance records:
6625


### Feature-Selection Frequency

Feature-selection frequency measures how consistently each feature was retained by the model across the five cross-validation folds.

A high selection frequency indicates that a feature repeatedly met the model's feature-selection criterion across different training subsets.

This provides an estimate of feature stability and helps distinguish consistently selected features from fold-specific features.

In [14]:
# Count the number of folds in which each feature was selected.
selection_frequency = (
    feature_importance_df
    .groupby(["Model", "Feature"])["Fold"]
    .nunique()
    .reset_index(name="Selection_Count")
)

# Convert the selection count to a proportion of the five folds.
selection_frequency["Selection_Frequency"] = (
    selection_frequency["Selection_Count"] / 5
)

selection_frequency.head()

,Model,Feature,Selection_Count,Selection_Frequency
0,LogReg_Gene,11715380_s_at,5,1.0
1,LogReg_Gene,11715670_a_at,2,0.4
2,LogReg_Gene,11715671_x_at,2,0.4
3,LogReg_Gene,11715946_a_at,1,0.2
4,LogReg_Gene,11716395_a_at,5,1.0


### Aggregate Feature Importance

Feature importance is summarized across the folds in which each feature was selected.

For Logistic Regression, the signed coefficient is retained to indicate the direction of association with the predicted class, while the absolute coefficient magnitude is used to assess importance.

For Random Forest, the model-provided feature importance is non-negative and is summarized directly.

Importance is therefore interpreted alongside selection frequency rather than as a standalone ranking.

In [15]:
# Calculate the absolute importance while retaining the original
# signed importance for Logistic Regression interpretation.
feature_importance_df["Absolute_Importance"] = (
    feature_importance_df["Importance"].abs()
)

# Summarize feature importance across the folds in which each
# feature was selected.
importance_summary = (
    feature_importance_df
    .groupby(["Model", "Feature"])
    .agg(
        Mean_Importance=("Importance", "mean"),
        Mean_Absolute_Importance=("Absolute_Importance", "mean"),
        Importance_SD=("Importance", "std"),
        Selection_Count=("Fold", "nunique")
    )
    .reset_index()
)

# Calculate the proportion of folds in which each feature was selected.
importance_summary["Selection_Frequency"] = (
    importance_summary["Selection_Count"] / 5
)

importance_summary.head()

,Model,Feature,Mean_Importance,Mean_Absolute_Importance,Importance_SD,Selection_Count,Selection_Frequency
0,LogReg_Gene,11715380_s_at,0.129667,0.129667,0.031647,5,1.0
1,LogReg_Gene,11715670_a_at,0.191629,0.191629,0.003393,2,0.4
2,LogReg_Gene,11715671_x_at,0.187880,0.187880,0.002402,2,0.4
3,LogReg_Gene,11715946_a_at,0.146438,0.146438,NaN,1,0.2
4,LogReg_Gene,11716395_a_at,0.123205,0.123205,0.012893,5,1.0


### Identify Stable and Important Features

Features selected in most cross-validation folds are considered more stable.

For interpretation, features selected in at least four of the five folds are retained as consistently selected features and ranked according to their mean absolute importance.

This approach prioritizes features that demonstrate both reproducible selection and substantial model contribution.

In [16]:
# Retain features selected in at least four of the five folds.
stable_features = importance_summary[
    importance_summary["Selection_Count"] >= 4
].copy()

# Rank stable features by their mean absolute importance.
stable_features = (
    stable_features
    .sort_values(
        ["Model", "Mean_Absolute_Importance"],
        ascending=[True, False]
    )
)

stable_features.head(20)

,Model,Feature,Mean_Importance,Mean_Absolute_Importance,Importance_SD,Selection_Count,Selection_Frequency
55,LogReg_Gene,11733889_a_at,-0.309026,0.309026,0.028470,5,1.0
63,LogReg_Gene,11737486_s_at,-0.267664,0.267664,0.021924,5,1.0
30,LogReg_Gene,11723945_s_at,0.216798,0.216798,0.033449,5,1.0
52,LogReg_Gene,11733087_a_at,0.208336,0.208336,0.023610,5,1.0
81,LogReg_Gene,11754183_s_at,-0.189258,0.189258,0.010808,5,1.0
21,LogReg_Gene,11722150_a_at,-0.180935,0.180935,0.021380,5,1.0
82,LogReg_Gene,11755612_s_at,-0.174229,0.174229,0.009893,5,1.0
49,LogReg_Gene,11730732_at,0.171187,0.171187,0.024631,5,1.0
39,LogReg_Gene,11726871_s_at,0.169350,0.169350,0.029196,5,1.0
64,LogReg_Gene,11739025_a_at,-0.167760,0.167760,0.007564,5,1.0


### Top Stable Features

The most stable features are ranked separately for each model.

The ranking is based on mean absolute model importance among features selected in at least four of the five cross-validation folds.

In [17]:
# Display the top 10 stable features for each model.
top_features = (
    stable_features
    .groupby("Model")
    .head(10)
)

top_features

,Model,Feature,Mean_Importance,Mean_Absolute_Importance,Importance_SD,Selection_Count,Selection_Frequency
55,LogReg_Gene,11733889_a_at,-0.309026,0.309026,0.028470,5,1.0
63,LogReg_Gene,11737486_s_at,-0.267664,0.267664,0.021924,5,1.0
30,LogReg_Gene,11723945_s_at,0.216798,0.216798,0.033449,5,1.0
52,LogReg_Gene,11733087_a_at,0.208336,0.208336,0.023610,5,1.0
81,LogReg_Gene,11754183_s_at,-0.189258,0.189258,0.010808,5,1.0
21,LogReg_Gene,11722150_a_at,-0.180935,0.180935,0.021380,5,1.0
82,LogReg_Gene,11755612_s_at,-0.174229,0.174229,0.009893,5,1.0
49,LogReg_Gene,11730732_at,0.171187,0.171187,0.024631,5,1.0
39,LogReg_Gene,11726871_s_at,0.169350,0.169350,0.029196,5,1.0
64,LogReg_Gene,11739025_a_at,-0.167760,0.167760,0.007564,5,1.0


### Visualize Top Stable Features

The top stable features are visualized to show their relative contribution to each final model.

Only features meeting the stability criterion are included.

In [ ]:
# Generate a horizontal bar plot of the top stable features for each model.
for model_name in stable_features["Model"].unique():

    plot_data = (
        stable_features[stable_features["Model"] == model_name]
        .head(10)
        .sort_values("Mean_Absolute_Importance")
    )

    plt.figure(figsize=(8, 5))

    plt.barh(
        plot_data["Feature"],
        plot_data["Mean_Absolute_Importance"]
    )

    plt.xlabel("Mean Absolute Importance")
    plt.ylabel("Feature")
    plt.title(f"Top Stable Features - {model_name}")

    plt.tight_layout()

    plt.savefig(
        f"../Results/Plots/top_features_{model_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

### Feature Agreement Between Models

Feature overlap between Logistic Regression and Random Forest is examined to identify features that contribute consistently across different modelling approaches.

Features identified by both models provide additional evidence of model-independent predictive relevance within the dataset.

In [18]:
# Identify stable features for each model.
logreg_gene_features = set(
    stable_features.loc[
        stable_features["Model"] == "LogReg_Gene",
        "Feature"
    ]
)

rf_gene_features = set(
    stable_features.loc[
        stable_features["Model"] == "RF_Gene",
        "Feature"
    ]
)

logreg_mirna_features = set(
    stable_features.loc[
        stable_features["Model"] == "LogReg_miRNA",
        "Feature"
    ]
)

rf_mirna_features = set(
    stable_features.loc[
        stable_features["Model"] == "RF_miRNA",
        "Feature"
    ]
)

# Identify features shared by both modelling approaches.
shared_gene_features = sorted(
    logreg_gene_features.intersection(rf_gene_features)
)

shared_mirna_features = sorted(
    logreg_mirna_features.intersection(rf_mirna_features)
)

print("Shared stable gene features:")
print(shared_gene_features)

print("\nShared stable miRNA features:")
print(shared_mirna_features)

Shared stable gene features:
['11719368_s_at', '11722150_a_at', '11723945_s_at', '11729059_x_at', '11730732_at', '11733088_at', '11736610_a_at', '11736611_s_at', '11736612_x_at', '11754183_s_at', '11755612_s_at', '11758454_s_at', '11758776_s_at']

Shared stable miRNA features:
['14qII-14_st', '14qII-14_x_st', '14qII-1_st', '14qII-1_x_st', '14qII-22_x_st', '14qII-23_x_st', '14qII-26_st', '14qII-26_x_st', '14qII-3_st', '14qII-3_x_st', 'age-miR-127_st', 'age-miR-20_st', 'age-miR-222_st', 'bta-miR-127_st', 'bta-miR-146b_st', 'bta-miR-181a_st', 'bta-miR-195_st', 'bta-miR-222_st', 'bta-miR-361_st', 'bta-miR-379_st', 'bta-miR-382_st', 'bta-miR-432_st', 'bta-miR-487b_st', 'bta-miR-493_st', 'bta-miR-494_st', 'cfa-miR-127_st', 'cfa-miR-134_st', 'cfa-miR-140_st', 'cfa-miR-181a_st', 'cfa-miR-195_st', 'cfa-miR-19b_st', 'cfa-miR-222_st', 'cfa-miR-361_st', 'cfa-miR-379_st', 'cfa-miR-432_st', 'cfa-miR-487b_st', 'cin-miR-4020b-5p_st', 'cre-miR1171_st', 'dre-miR-140-star_st', 'dre-miR-181a_st', 'dre-miR

### Summary

The feature-importance analysis provides an interpretation of the features contributing to the trained classification models.

- Feature selection frequency was assessed across cross-validation folds to identify features that were repeatedly selected by the models.
- Model-specific importance scores were examined alongside selection frequency to distinguish consistently selected features from features with stronger but less stable contributions.
- Gene expression and miRNA models were evaluated separately, reflecting the different feature spaces and selected feature-set sizes used during model development.
- The resulting feature rankings describe the features most consistently associated with model predictions within the available cohort.

These results provide model-level interpretability and support assessment of feature stability. The reported features should not be interpreted as independently validated biological biomarkers, particularly given the small sample size and high-dimensional nature of the datasets.